In [11]:
import calendar
import json
import re
from datetime import date, datetime

from bs4 import BeautifulSoup
from rich.console import Console
from rich.table import Table

with open("dumps.html") as file:
    soup = BeautifulSoup(file, "lxml")

**Extraction**

Within a for loop:
Extract the `<li>` tag which contains PSET data within it
- the `<a>` tag has the PSET name
- the `<span data-moment='LLLL'>` has the submitted date

Create an empty list to fill with pset data : 
- `pset_list`

Using regex and find within the `<li>` element to retrieve:
- `pset_name` = shirtificate, type = <class 'str'>
- `pset_date` = 2026-08-12 08:23:14Z, type = <class 'str'>

The date includes time, so we split it
- `pset_str_date` = 2026-08-12, type = <class 'str'>
- `pset_str_time` = 08:23:14Z, type = <class 'str'>

At this point we don't need the time, so we convert date to datetime format
- `pset_dt_date` = 2026-08-12, type = <class 'datetime.date'>

Construct a dictionary `pset_dict` with 
- keys : `pset_name`, `pset_date`
- value(type) : str, datetime.date
`{'pset_name': 'shirtificate', 'pset_date': datetime.date(2026, 8, 12)}`

Append to `pset_list`, the constructed `pset_dict` dictionary
- pset_list
[
  {
    "pset_name": "less",
    "pset_date": "2024-01-12"
  },
 .
 .
 .
  {
    "pset_name": "shirtificate",
    "pset_date": "2026-08-12"
  }
]

Reverse the list, so earlier pset submissions come first


In [ ]:
pset_list = []
# create pset date list so i can check during calendar creation
# if date in pset date list, then add str modifier
# if already modified then modify again to indicate more pset submission

# def get_ps_name():

for item in soup.find_all("li"):
    tag = item.a.text.strip()
    pset_name = re.sub(r".*/", "", tag)
    # print(f"pset_name = {pset_name}, type = {type(pset_name)}")
    pset_date = item.find(attrs={"data-moment": "LLLL"}).text.strip()
    # print(f"pset_date = {pset_date}, type = {type(pset_date)}")
    pset_str_date, pset_str_time = pset_date.split(" ")
    # print(f"pset_str_date = {pset_str_date}, type = {type(pset_str_date)}")
    # print(f"pset_str_time = {pset_str_time}, type = {type(pset_str_time)}")
    pset_dt_date = datetime.strptime(pset_str_date, "%Y-%m-%d").date()
    #  <class 'datetime.date'>
    print(f"pset_dt_date = {pset_dt_date}, type = {type(pset_dt_date)}")
    # print("----")
    pset_dict = {"pset_name": pset_name, "pset_date": pset_dt_date}
    # print(pset_dict)
    pset_list.append(pset_dict)
    # print(pset_list)
    # break

pset_list.reverse()

print(json.dumps(pset_list, indent=2, default=str))

Now that I have retrieved pset data in a list of dictionaries, I need to generate the calendar.

I want calendars for each month that has a pset submission in it
- ignore months without pset submissions. 
- Each month should appear only once
- It should span multiple years if needed. 

I created a blank list to populate with month/year pairs for the calendar months I need to generate
- year_month_list = []

Loop through my psets, 
for `pset` in `pset_list`

and create a (year, month)  tuple
- pset_year_month = (2024, 1), type = <class 'tuple'>

Append the list with the (year, month) tuple if its not in the list already
pset["pset_date"].year = 2024, type = <class 'int'>
pset["pset_date"].month = 1, type = <class 'int'>

year_month_list =
[(2024, 1), (2024, 4), (2024, 5), ....., (2026, 7), (2026, 8)]

In [13]:
year_month_list = []


for pset in pset_list:
    pset_year_month = (pset["pset_date"].year, pset["pset_date"].month)
    if pset_year_month not in year_month_list:
        year_month_list.append((pset["pset_date"].year, pset["pset_date"].month))
    # break

print(year_month_list)

[(2024, 1), (2024, 4), (2024, 5), (2024, 6), (2024, 7), (2024, 8), (2025, 6), (2026, 6), (2026, 7), (2026, 8)]


Created a calendar object with firstweek = Sunday
Created a list `cal_rows` which will be used to construct our final calendar.
`cal_rows` is a list of lists.

Each list within is a row in the generated calendar, which is generated from cal.monthdayscalendar, and includes an extra list item before and after the day items.
Each day in this sub list, is class int
do a condition check on the date

These extra rows will be used to indicate the Month name and the Year, respectively.
Before adding the rows, we will iterate through each day in the list, and check whether that day exists in pset_list.get("pset_date") and if it does, we will modify it with formatting for richtable to color green

In [14]:
cal = calendar.Calendar(firstweekday=calendar.SUNDAY)
color = "spring_green2"
cal_rows = []
for year_month in year_month_list:
    year, month = year_month  #  int, int
    # print(type(year), type(month))
    # print(year, month)
    for week in cal.monthdayscalendar(year, month):
        # print(week) #  each item in the week list, is class <int>
        # run a condition here for checking
        # if date(year, month, listitem) in
        for i, day in enumerate(week):
            for j, pset in enumerate(pset_list):
                if day != 0 and pset.get("pset_date") == date(year, month, day):
                    week[i] = f"[{color}]{week[i]}[/{color}]"
                    del pset_list[j]
        if 1 in week:
            week.insert(0, calendar.month_abbr[month].upper())
            week.append(year)
        else:
            week.insert(0, " ")
            week.append(" ")
        # print(week)
        cal_rows.append(week)
        # break
    # break

print(cal_rows)

[['JAN', 0, 1, 2, 3, 4, 5, 6, 2024], [' ', 7, 8, 9, 10, 11, '[spring_green2]12[/spring_green2]', '[spring_green2]13[/spring_green2]', ' '], [' ', 14, '[spring_green2]15[/spring_green2]', 16, 17, 18, 19, 20, ' '], [' ', 21, 22, 23, 24, 25, 26, 27, ' '], [' ', 28, 29, 30, 31, 0, 0, 0, ' '], ['APR', 0, 1, 2, 3, 4, 5, 6, 2024], [' ', 7, 8, 9, 10, 11, 12, 13, ' '], [' ', 14, 15, '[spring_green2]16[/spring_green2]', 17, 18, 19, 20, ' '], [' ', '[spring_green2]21[/spring_green2]', 22, 23, 24, '[spring_green2]25[/spring_green2]', 26, 27, ' '], [' ', 28, 29, 30, 0, 0, 0, 0, ' '], ['MAY', 0, 0, 0, 1, 2, 3, 4, 2024], [' ', 5, 6, 7, 8, '[spring_green2][spring_green2][spring_green2]9[/spring_green2][/spring_green2][/spring_green2]', 10, '[spring_green2]11[/spring_green2]', ' '], [' ', 12, '[spring_green2]13[/spring_green2]', '[spring_green2]14[/spring_green2]', 15, 16, 17, 18, ' '], [' ', 19, 20, 21, 22, 23, 24, 25, ' '], [' ', 26, 27, 28, 29, 30, 31, 0, ' '], ['JUN', 0, 0, 0, 0, 0, 0, 1, 2024], ['

i need to iterate through the cal_rows list, and convert them to strings as table is not parsing them. unable to get render width it says

then iterate through the new list and add rows to the table with each list index per row

In [15]:
console = Console()
table = Table()

days = ["MNTH", "SU", "MO", "TU", "WE", "TH", "FR", "SA", "YEAR"]
# cal_rows.insert(0, days)

str_rows = []

for row in cal_rows:
    ind_row = []
    for item in row:
        ind_row.append(str(item))
    str_rows.append(ind_row)


print(str_rows)

table = Table("MNTH", "SU", "MO", "TU", "WE", "TH", "FR", "SA", "YEAR", title="CS50p")

for row in str_rows:
    table.add_row(
        row[0], row[1], row[2], row[3], row[4], row[5], row[6], row[7], row[8]
    )


console.print(table)

[['JAN', '0', '1', '2', '3', '4', '5', '6', '2024'], [' ', '7', '8', '9', '10', '11', '[spring_green2]12[/spring_green2]', '[spring_green2]13[/spring_green2]', ' '], [' ', '14', '[spring_green2]15[/spring_green2]', '16', '17', '18', '19', '20', ' '], [' ', '21', '22', '23', '24', '25', '26', '27', ' '], [' ', '28', '29', '30', '31', '0', '0', '0', ' '], ['APR', '0', '1', '2', '3', '4', '5', '6', '2024'], [' ', '7', '8', '9', '10', '11', '12', '13', ' '], [' ', '14', '15', '[spring_green2]16[/spring_green2]', '17', '18', '19', '20', ' '], [' ', '[spring_green2]21[/spring_green2]', '22', '23', '24', '[spring_green2]25[/spring_green2]', '26', '27', ' '], [' ', '28', '29', '30', '0', '0', '0', '0', ' '], ['MAY', '0', '0', '0', '1', '2', '3', '4', '2024'], [' ', '5', '6', '7', '8', '[spring_green2][spring_green2][spring_green2]9[/spring_green2][/spring_green2][/spring_green2]', '10', '[spring_green2]11[/spring_green2]', ' '], [' ', '12', '[spring_green2]13[/spring_green2]', '[spring_green2]

                      CS50p                       
┏━━━━━━┳━━━━┳━━━━┳━━━━┳━━━━┳━━━━┳━━━━┳━━━━┳━━━━━━┓
┃ MNTH ┃ SU ┃ MO ┃ TU ┃ WE ┃ TH ┃ FR ┃ SA ┃ YEAR ┃
┡━━━━━━╇━━━━╇━━━━╇━━━━╇━━━━╇━━━━╇━━━━╇━━━━╇━━━━━━┩
│ JAN  │ 0  │ 1  │ 2  │ 3  │ 4  │ 5  │ 6  │ 2024 │
│      │ 7  │ 8  │ 9  │ 10 │ 11 │ 12 │ 13 │      │
│      │ 14 │ 15 │ 16 │ 17 │ 18 │ 19 │ 20 │      │
│      │ 21 │ 22 │ 23 │ 24 │ 25 │ 26 │ 27 │      │
│      │ 28 │ 29 │ 30 │ 31 │ 0  │ 0  │ 0  │      │
│ APR  │ 0  │ 1  │ 2  │ 3  │ 4  │ 5  │ 6  │ 2024 │
│      │ 7  │ 8  │ 9  │ 10 │ 11 │ 12 │ 13 │      │
│      │ 14 │ 15 │ 16 │ 17 │ 18 │ 19 │ 20 │      │
│      │ 21 │ 22 │ 23 │ 24 │ 25 │ 26 │ 27 │      │
│      │ 28 │ 29 │ 30 │ 0  │ 0  │ 0  │ 0  │      │
│ MAY  │ 0  │ 0  │ 0  │ 1  │ 2  │ 3  │ 4  │ 2024 │
│      │ 5  │ 6  │ 7  │ 8  │ 9  │ 10 │ 11 │      │
│      │ 12 │ 13 │ 14 │ 15 │ 16 │ 17 │ 18 │      │
│      │ 19 │ 20 │ 21 │ 22 │ 23 │ 24 │ 25 │      │
│      │ 26 │ 27 │ 28 │ 29 │ 30 │ 31 │ 0  │      │
│ JUN  │ 0  │ 0  │ 0  │ 0  │ 0  │ 0  │ 1  │ 2024 │
│      │ 2  │ 3  │ 4  │ 5  │ 6  │ 7  │ 8  │      │
│      │ 9  │ 10 │ 11 │ 12 │ 13 │ 14 │ 15 │      │
│      │ 16 │ 17 │ 18 │ 19 │ 20 │ 21 │ 22 │      │
│      │ 23 │ 24 │ 25 │ 26 │ 27 │ 28 │ 29 │      │
│      │ 30 │ 0  │ 0  │ 0  │ 0  │ 0  │ 0  │      │
│ JUL  │ 0  │ 1  │ 2  │ 3  │ 4  │ 5  │ 6  │ 2024 │
│      │ 7  │ 8  │ 9  │ 10 │ 11 │ 12 │ 13 │      │
│      │ 14 │ 15 │ 16 │ 17 │ 18 │ 19 │ 20 │      │
│      │ 21 │ 22 │ 23 │ 24 │ 25 │ 26 │ 27 │      │
│      │ 28 │ 29 │ 30 │ 31 │ 0  │ 0  │ 0  │      │
│ AUG  │ 0  │ 0  │ 0  │ 0  │ 1  │ 2  │ 3  │ 2024 │
│      │ 4  │ 5  │ 6  │ 7  │ 8  │ 9  │ 10 │      │
│      │ 11 │ 12 │ 13 │ 14 │ 15 │ 16 │ 17 │      │
│      │ 18 │ 19 │ 20 │ 21 │ 22 │ 23 │ 24 │      │
│      │ 25 │ 26 │ 27 │ 28 │ 29 │ 30 │ 31 │      │
│ JUN  │ 1  │ 2  │ 3  │ 4  │ 5  │ 6  │ 7  │ 2025 │
│      │ 8  │ 9  │ 10 │ 11 │ 12 │ 13 │ 14 │      │
│      │ 15 │ 16 │ 17 │ 18 │ 19 │ 20 │ 21 │      │
│      │ 22 │ 23 │ 24 │ 25 │ 26 │ 27 │ 28 │      │
│      │ 29 │ 30 │ 0  │ 0  │ 0  │ 0  │ 0  │      │
│ JUN  │ 0  │ 1  │ 2  │ 3  │ 4  │ 5  │ 6  │ 2026 │
│      │ 7  │ 8  │ 9  │ 10 │ 11 │ 12 │ 13 │      │
│      │ 14 │ 15 │ 16 │ 17 │ 18 │ 19 │ 20 │      │
│      │ 21 │ 22 │ 23 │ 24 │ 25 │ 26 │ 27 │      │
│      │ 28 │ 29 │ 30 │ 0  │ 0  │ 0  │ 0  │      │
│ JUL  │ 0  │ 0  │ 0  │ 1  │ 2  │ 3  │ 4  │ 2026 │
│      │ 5  │ 6  │ 7  │ 8  │ 9  │ 10 │ 11 │      │
│      │ 12 │ 13 │ 14 │ 15 │ 16 │ 17 │ 18 │      │
│      │ 19 │ 20 │ 21 │ 22 │ 23 │ 24 │ 25 │      │
│      │ 26 │ 27 │ 28 │ 29 │ 30 │ 31 │ 0  │      │
│ AUG  │ 0  │ 0  │ 0  │ 0  │ 0  │ 0  │ 1  │ 2026 │
│      │ 2  │ 3  │ 4  │ 5  │ 6  │ 7  │ 8  │      │
│      │ 9  │ 10 │ 11 │ 12 │ 13 │ 14 │ 15 │      │
│      │ 16 │ 17 │ 18 │ 19 │ 20 │ 21 │ 22 │      │
│      │ 23 │ 24 │ 25 │ 26 │ 27 │ 28 │ 29 │      │
│      │ 30 │ 31 │ 0  │ 0  │ 0  │ 0  │ 0  │      │
└──────┴────┴────┴────┴────┴────┴────┴────┴──────┘